# moviepy — Final composite

Composes the polls-tutorial demo into a single mp4 from:

- per-scene narration: `audio/scene_NN.mp3` (committed; rendered by `narrator.ipynb`)
- per-scene Manim clips: `media/videos/source/480p15/<SceneClass>.mp4` (gitignored; rendered by `manim.ipynb` — re-run that notebook to regenerate)

**Per-scene strategy**:
1. Load the narration `mp3`. Its duration is the canonical scene length.
2. Concatenate that scene's Manim sub-clips back-to-back.
3. If Manim is shorter than the narration (currently the case for every scene), hold the last frame for the remainder.
4. Attach the narration as the scene's audio track.

The 9 scene clips are then concatenated into `out/polls_demo.mp4`.

`out/` is gitignored — re-run this notebook to regenerate the final video on a fresh checkout.


In [ ]:
from pathlib import Path

from moviepy import (
    AudioFileClip,
    ImageClip,
    VideoFileClip,
    concatenate_videoclips,
)

# Notebook lives in source/, so paths are relative to that.
AUDIO_DIR = Path("audio")
VIDEO_DIR = Path("media/videos/source/480p15")
OUT_DIR = Path("out")
OUT_DIR.mkdir(exist_ok=True)
OUT_PATH = OUT_DIR / "polls_demo.mp4"

assert AUDIO_DIR.exists(), f"missing {AUDIO_DIR.resolve()}"
assert VIDEO_DIR.exists(), (
    f"missing {VIDEO_DIR.resolve()} — re-run manim.ipynb to render the scene mp4s"
)

print(f"audio dir: {AUDIO_DIR.resolve()}")
print(f"video dir: {VIDEO_DIR.resolve()}")
print(f"out      : {OUT_PATH.resolve()}")


In [ ]:
# Per-scene assets. Order matches the conte.md storyboard.
SCENES = [
    {"id": 1, "audio": "scene_01.mp3", "videos": ["Scene1Opening.mp4"]},
    {"id": 2, "audio": "scene_02.mp3", "videos": ["Scene2Reinhardt.mp4"]},
    {"id": 3, "audio": "scene_03.mp3", "videos": ["Scene3aTree.mp4", "Scene3bMakeTasks.mp4"]},
    {"id": 4, "audio": "scene_04.mp3", "videos": ["Scene4aModelMacro.mp4", "Scene4bQuestionChoiceERD.mp4"]},
    {"id": 5, "audio": "scene_05.mp3", "videos": ["Scene5aServerFnRPC.mp4", "Scene5bUseActionCycle.mp4"]},
    {"id": 6, "audio": "scene_06.mp3", "videos": ["Scene6aFormMacro.mp4"]},
    {"id": 7, "audio": "scene_07.mp3", "videos": ["Scene7aAdminRegister.mp4"]},
    {"id": 8, "audio": "scene_08.mp3", "videos": ["Scene8aDIProviders.mp4", "Scene8bGuardChain.mp4"]},
    {"id": 9, "audio": "scene_09.mp3", "videos": ["Scene9Closing.mp4"]},
]

# Tail silence appended to each scene. Gives the listener a beat between
# scenes and is the easiest knob to stretch toward the 13:30 conte.md target.
TAIL_SILENCE = 0.6  # seconds


In [ ]:
def build_scene(scene):
    """Compose one scene: Manim sub-clips synced to narration, freeze-padded if short."""
    audio = AudioFileClip(str(AUDIO_DIR / scene["audio"]))
    target = audio.duration + TAIL_SILENCE

    video_clips = [VideoFileClip(str(VIDEO_DIR / v)) for v in scene["videos"]]
    combined = (
        video_clips[0]
        if len(video_clips) == 1
        else concatenate_videoclips(video_clips, method="chain")
    )

    if combined.duration + 1e-3 < target:
        # Hold the last frame to fill the remaining time.
        last_frame = combined.get_frame(max(0.0, combined.duration - 0.05))
        freeze = ImageClip(last_frame).with_duration(target - combined.duration)
        freeze.fps = combined.fps
        combined = concatenate_videoclips([combined, freeze], method="chain")
    elif combined.duration > target:
        combined = combined.subclipped(0, target)

    return combined.with_duration(target).with_audio(audio)


# Pre-flight duration plan.
print(f"{'SCENE':>5} {'AUDIO':>10} {'MANIM':>10} {'OUT':>10}")
total = 0.0
for scene in SCENES:
    a = AudioFileClip(str(AUDIO_DIR / scene["audio"])).duration
    v = sum(VideoFileClip(str(VIDEO_DIR / vn)).duration for vn in scene["videos"])
    out = a + TAIL_SILENCE
    total += out
    print(f"   {scene['id']:>2}  {a:8.2f}s  {v:8.2f}s  {out:8.2f}s")
print(f"TOTAL: {total:.2f}s  ({total / 60:.2f} min)")


In [ ]:
scene_clips = [build_scene(s) for s in SCENES]
final = concatenate_videoclips(scene_clips, method="chain")

print(f"Final duration: {final.duration:.2f}s ({final.duration / 60:.2f} min)")
print(f"Writing {OUT_PATH} ...")

final.write_videofile(
    str(OUT_PATH),
    fps=15,
    codec="libx264",
    audio_codec="aac",
    threads=4,
    preset="medium",
)

print("Done.")


In [ ]:
# Sanity check the final mp4.
clip = VideoFileClip(str(OUT_PATH))
size_mb = OUT_PATH.stat().st_size / 1024 / 1024
print(f"path      : {OUT_PATH}")
print(f"size      : {size_mb:.1f} MB")
print(f"duration  : {clip.duration:.2f}s ({clip.duration / 60:.2f} min)")
print(f"fps       : {clip.fps}")
print(f"resolution: {clip.size}")
clip.close()
